In [1]:
import collections
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup

# Define file extensions to ignore
SKIP_EXTENSIONS = (
    ".pdf",
    ".jpg",
    ".jpeg",
    ".png",
    ".gif",
    ".zip",
    ".docx",
    ".xlsx",
    ".mp4",
    ".svg",
    ".csv",
)


def fetch_and_extract_links(url, domain, session):
    """Fetches a single page using a shared HTTP session and returns newly found valid internal URLs."""
    try:
        response = session.get(url, timeout=5)

        # Ignore non-HTML pages quickly
        if "text/html" not in response.headers.get("Content-Type", ""):
            return []

        # Parse with lxml for faster extraction (falls back to html.parser if lxml isn't installed)
        try:
            soup = BeautifulSoup(response.text, "lxml")
        except Exception:
            soup = BeautifulSoup(response.text, "html.parser")

        found_urls = set()
        for anchor in soup.find_all("a", href=True):
            href = anchor["href"].strip()
            full_url = urljoin(url, href)
            parsed = urlparse(full_url)

            # Filter for internal domain links only
            if parsed.netloc == domain and parsed.scheme in ("http", "https"):
                clean_url = full_url.split("#")[0].rstrip("/")
                clean_path = parsed.path.lower()

                if not clean_path.endswith(SKIP_EXTENSIONS):
                    found_urls.add(clean_url)

        return list(found_urls)
    except Exception:
        return []


def discover_all_urls(start_url, max_workers=20):
    """Crawls a domain using parallel thread execution and HTTP session pooling."""
    domain = urlparse(start_url).netloc
    visited = set()
    to_visit = {start_url.split("#")[0].rstrip("/")}

    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
            " (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
        )
    })

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        while to_visit:
            # Take a batch of unvisited URLs
            current_batch = list(to_visit)
            to_visit.clear()

            # Mark batch as visited upfront to prevent duplicate scheduling
            for url in current_batch:
                visited.add(url)
                print(f"Scanning: {url}")

            # Submit batch jobs to thread pool
            futures = {
                executor.submit(
                    fetch_and_extract_links, url, domain, session
                ): url
                for url in current_batch
            }

            for future in as_completed(futures):
                new_urls = future.result()
                for n_url in new_urls:
                    if n_url not in visited:
                        to_visit.add(n_url)

    return sorted(list(visited))


if __name__ == "__main__":
    target_site = "https://psw.gov.pk"
    all_urls = discover_all_urls(target_site, max_workers=20)

    print(f"\nDiscovered {len(all_urls)} HTML URLs:")
    print(all_urls)

Scanning: https://psw.gov.pk
Scanning: https://psw.gov.pk/user-manuals
Scanning: https://psw.gov.pk/csr
Scanning: https://psw.gov.pk/privacy-policy?id=8
Scanning: https://psw.gov.pk/documents/tenders
Scanning: https://psw.gov.pk/ticket-desk
Scanning: https://psw.gov.pk/contact-us
Scanning: https://psw.gov.pk/our-partners
Scanning: https://psw.gov.pk/services/moi
Scanning: https://psw.gov.pk/psqca
Scanning: https://psw.gov.pk/utilities?service=clearance_data
Scanning: https://psw.gov.pk/weboc-to-psw
Scanning: https://psw.gov.pk/khadijah-program
Scanning: https://psw.gov.pk/utilities?service=vehicle_status
Scanning: https://psw.gov.pk/bra
Scanning: https://psw.gov.pk/utilities?service=duty_calculator
Scanning: https://psw.gov.pk/leadership
Scanning: https://psw.gov.pk/utilities?service=tariff_sros
Scanning: https://psw.gov.pk/documents/legal
Scanning: https://psw.gov.pk/trade-lab
Scanning: https://psw.gov.pk/our-history
Scanning: https://psw.gov.pk/about-psw
Scanning: https://psw.gov.pk/

In [2]:
print(len(all_urls))

236


In [3]:
from utils import (
    sanitize_and_generate_dynamic_folder,
    dynamic_session_extractor,
    download_single_asset
)

In [4]:
if __name__ == "__main__":
    print("=== INITIALIZING PARALLEL SERVER-ALIGNMENT BATCH PIPELINE ===")
    
    # 1. Clean and prepare all URLs into work units
    work_units = []
    for raw_url in all_urls:
        clean_url, session_workspace = sanitize_and_generate_dynamic_folder(raw_url)
        if clean_url:
            work_units.append((clean_url, session_workspace))

    total_tasks = len(work_units)
    print(f"Queue Size: {total_tasks} locations locked into memory.\n" + "=" * 60)

    # 2. Concurrently execute tasks
    max_workers = 15  # Adjust based on system resources and bandwidth
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit tasks and track original indices for progress logging
        future_to_task = {
            executor.submit(dynamic_session_extractor, clean_url, session_workspace): (idx, clean_url, session_workspace)
            for idx, (clean_url, session_workspace) in enumerate(work_units, start=1)
        }

        # Handle completion as tasks finish
        for future in as_completed(future_to_task):
            idx, clean_url, session_workspace = future_to_task[future]
            
            print(f"\n[{idx}/{total_tasks}] Container Workflow Completed")
            print(f"   ↳ Target URL: {clean_url}")
            print(f"   ↳ Destination Path: {session_workspace}")
            
            try:
                future.result()
            except Exception as e:
                print(f"   ↳ ✗ Execution Failed: {e}")

    print("\n" + "=" * 60 + "\n=== SYSTEM PIPELINE PROCESSING COMPLETE ===")

=== INITIALIZING PARALLEL SERVER-ALIGNMENT BATCH PIPELINE ===
Queue Size: 236 locations locked into memory.
   ✓ Extracted text -> psw_gov_pk_about-psw_6d6838.txt
   ✓ Extracted text -> psw_gov_pk_career-opportunities_page_2_7ef53a.txt
   ✓ Extracted text -> psw_gov_pk_contact-us_7e07c0.txt
   ✓ Extracted text -> psw_gov_pk_blogs_page_2_f721f6.txt
   ✓ Extracted text -> psw_gov_pk_aqd_149eca.txt
   ✓ Extracted text -> psw_gov_pk_7e8e39.txt
   ✓ Extracted text -> psw_gov_pk_career-opportunities_page_1_9394e7.txt
   ✓ Extracted text -> psw_gov_pk_dei-at-psw_584b21.txt
   ✓ Extracted text -> psw_gov_pk_career-opportunities_e87a7f.txt
   ✓ Extracted text -> psw_gov_pk_csr_5f3040.txt

[13/236] Container Workflow Completed
   ↳ Target URL: https://psw.gov.pk/contact-us
   ↳ Destination Path: downloads\psw_gov_pk_contact-us_7e07c0

[11/236] Container Workflow Completed
   ↳ Target URL: https://psw.gov.pk/career-opportunities?page=2
   ↳ Destination Path: downloads\psw_gov_pk_career-opportunit